In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from PIL import Image
import pandas as pd
from pathlib import Path
import os


# КОНСТАНТЫ И НАСТРОЙКИ
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Используем устройство: {device}")

# Пути к данным
DATA_DIR = Path("../data")  
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

# Константы нормализации (ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Гиперпараметры (из лучшего запуска)
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
NUM_EPOCHS = 30


# АУГМЕНТАЦИИ 
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Трансформации для валидации/теста (без аугментаций)
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# МОДЕЛЬ (ResNet18 с заморозкой)
def create_resnet18_frozen(num_classes=2):
    """Создаёт ResNet18 с замороженными весами (кроме последнего слоя)."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # Замораживаем все слои
    for param in model.parameters():
        param.requires_grad = False
    
    # Заменяем последний слой
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, num_classes)
    )
    
    return model


# ОБУЧЕНИЕ НА ВСЕХ ДАННЫХ
print("ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ НА ВСЕХ 40 ИЗОБРАЖЕНИЯХ")

# Загружаем полный датасет
full_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
train_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Создаём модель
model = create_resnet18_frozen(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()

# Оптимизируем последний слой
optimizer = optim.AdamW(model.fc.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# Обучаем
best_loss = float('inf')
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    scheduler.step(epoch_loss)
    
    # Сохраняем лучшую модель
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), "final_submission_model.pth")
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

print(f"\nФинальная модель сохранена: final_submission_model.pth")
print(f"Лучший Train Loss: {best_loss:.4f}")


# ЗАГРУЗКА ЛУЧШЕЙ МОДЕЛИ
print("ЗАГРУЗКА ЛУЧШЕЙ МОДЕЛИ")
model = create_resnet18_frozen(num_classes=2).to(device)
model.load_state_dict(torch.load("final_submission_model.pth", map_location=device))
model.eval()
print("Модель загружена и готова к инференсу")


# ИНФЕРЕНС НА ТЕСТЕ
print("ИНФЕРЕНС НА ТЕСТОВЫХ ДАННЫХ")
# Создаём тестовый датасет
class TestImageDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = Path(img_dir)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Получаем ID изображения
        img_id = str(self.df.iloc[idx]['id']).zfill(4)  # Дополняем до 4 цифр
        
        # Ищем файл (может быть .jpg или .png)
        img_path = self.img_dir / f"{img_id}.jpg"
        if not img_path.exists():
            img_path = self.img_dir / f"{img_id}.png"
        
        # Открываем изображение
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, img_id

test_dataset = TestImageDataset(
    csv_path=SUBMISSION_PATH,
    img_dir=TEST_DIR,
    transform=val_transforms  
)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Делаем предсказания
idx_to_class = {0: 'cleaned', 1: 'dirty'}  
predictions = []
image_ids = []

print("Делаем предсказания...")
with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        for img_id, pred_idx in zip(ids, predicted.cpu().numpy()):
            image_ids.append(img_id)
            predictions.append(idx_to_class[pred_idx])

print(f"Сделано {len(predictions)} предсказаний")

# СОЗДАНИЕ SUBMISSION.CSV
print("СОЗДАНИЕ SUBMISSION.CSV")
submission_df = pd.DataFrame({
    'id': image_ids,
    'label': predictions
})

# Сохраняем
submission_path = DATA_DIR / "submission_resnet18_final.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Submission сохранён: {submission_path}")
print(f"\nРаспределение классов:")
print(submission_df['label'].value_counts())
print(f"\nПервые 10 строк:")
print(submission_df.head(10))

# Проверяем количество
total = len(submission_df)
cleaned = (submission_df['label'] == 'cleaned').sum()
dirty = (submission_df['label'] == 'dirty').sum()

print(f"\nСтатистика:")
print(f"   Всего изображений: {total}")
print(f"   Cleaned: {cleaned} ({cleaned/total*100:.1f}%)")
print(f"   Dirty: {dirty} ({dirty/total*100:.1f}%)")

Используем устройство: mps
ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ НА ВСЕХ 40 ИЗОБРАЖЕНИЯХ
Epoch 05 | Loss: 0.5952 | Acc: 0.7250
Epoch 10 | Loss: 0.5769 | Acc: 0.6750
Epoch 15 | Loss: 0.4347 | Acc: 0.7750
Epoch 20 | Loss: 0.4987 | Acc: 0.7750
Epoch 25 | Loss: 0.3772 | Acc: 0.8750
Epoch 30 | Loss: 0.3518 | Acc: 0.9000

Финальная модель сохранена: final_submission_model.pth
Лучший Train Loss: 0.3300
ЗАГРУЗКА ЛУЧШЕЙ МОДЕЛИ
Модель загружена и готова к инференсу
ИНФЕРЕНС НА ТЕСТОВЫХ ДАННЫХ
Делаем предсказания...
Сделано 744 предсказаний
СОЗДАНИЕ SUBMISSION.CSV
Submission сохранён: ../data/submission_resnet18_final.csv

Распределение классов:
label
dirty      696
cleaned     48
Name: count, dtype: int64

Первые 10 строк:
     id  label
0  0000  dirty
1  0001  dirty
2  0002  dirty
3  0003  dirty
4  0004  dirty
5  0005  dirty
6  0006  dirty
7  0007  dirty
8  0008  dirty
9  0009  dirty

Статистика:
   Всего изображений: 744
   Cleaned: 48 (6.5%)
   Dirty: 696 (93.5%)
